In [4]:
# ============================================================
# OPTIMIZED LSTM LANGUAGE MODEL - TINY SHAKESPEARE
# GOOGLE COLAB VERSION
# ============================================================

import math
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os
import kagglehub

# ============================================================
# GPU OPTIMIZATION
# ============================================================

torch.backends.cudnn.benchmark = True

# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42
torch.manual_seed(SEED)

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ============================================================
# CONFIGURATION (OPTIMIZED)
# ============================================================

BLOCK_SIZE = 64
BATCH_SIZE = 256
EPOCHS = 5
LR = 3e-4
GRAD_CLIP = 1.0
PERPLEXITY_THRESHOLD = 50

# ============================================================
# DOWNLOAD DATASET
# ============================================================

def download_dataset():

    path = kagglehub.dataset_download("kaushaltiwari/tiny-shakespeare")

    for file in os.listdir(path):
        if file.endswith(".txt"):
            return os.path.join(path, file)

# ============================================================
# LOAD DATA
# ============================================================

def load_data(dataset_path):

    with open(dataset_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    lines = lines[:40000]

    train_lines = lines[:30000]
    val_lines = lines[30000:32000]
    test_lines = lines[32000:40000]

    return train_lines, val_lines, test_lines

# ============================================================
# VOCAB
# ============================================================

def build_vocab(train_lines):

    text = "".join(train_lines)

    chars = sorted(list(set(text)))

    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}

    vocab_size = len(chars)

    return stoi, itos, vocab_size

# ============================================================
# ENCODE DATA
# ============================================================

def encode_data(lines, stoi):

    text = "".join(lines)

    encoded = torch.tensor([stoi[c] for c in text], dtype=torch.long)

    return encoded

# ============================================================
# DATASET
# ============================================================

class CharDataset(Dataset):

    def __init__(self, data, block_size):

        self.data = data
        self.block_size = block_size

    def __len__(self):

        return len(self.data) - self.block_size

    def __getitem__(self, idx):

        x = self.data[idx:idx + self.block_size]
        y = self.data[idx + 1:idx + self.block_size + 1]

        return x, y

# ============================================================
# FAST LSTM MODEL
# ============================================================

class LSTM(nn.Module):

    def __init__(self, vocab_size, block_size, device,
                 embed_dim=128,
                 hidden_dim=256,
                 num_layers=1):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):

        x = self.embedding(x)

        out, _ = self.lstm(x)

        logits = self.fc(out)

        return logits

# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(model, dataloader, optimizer, criterion):

    model.train()

    total_loss = 0
    total_tokens = 0

    start = time.time()

    for x, y in dataloader:

        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()

        total_loss += loss.item()

        total_tokens += x.numel()

    epoch_time = time.time() - start

    tokens_per_second = total_tokens / epoch_time

    return total_loss / len(dataloader), epoch_time, tokens_per_second

# ============================================================
# EVALUATION
# ============================================================

def evaluate(model, dataloader, criterion):

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for x, y in dataloader:

            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            logits = model(x)

            loss = criterion(
                logits.view(-1, logits.size(-1)),
                y.view(-1)
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)

# ============================================================
# PARAMETER COUNT
# ============================================================

def count_parameters(model):

    total = sum(p.numel() for p in model.parameters())

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return total, trainable

# ============================================================
# MAIN PIPELINE
# ============================================================

def main():

    dataset_path = download_dataset()

    train_lines, val_lines, test_lines = load_data(dataset_path)

    stoi, itos, vocab_size = build_vocab(train_lines)

    print("Vocabulary size:", vocab_size)

    train_data = encode_data(train_lines, stoi)
    val_data = encode_data(val_lines, stoi)
    test_data = encode_data(test_lines, stoi)

    train_dataset = CharDataset(train_data, BLOCK_SIZE)
    val_dataset = CharDataset(val_data, BLOCK_SIZE)
    test_dataset = CharDataset(test_data, BLOCK_SIZE)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        num_workers=2,
        pin_memory=True
    )

    model = LSTM(vocab_size, BLOCK_SIZE, DEVICE).to(DEVICE)

    total_params, trainable_params = count_parameters(model)

    print("Parameters:", total_params)

    flops_per_forward = 2 * total_params * BLOCK_SIZE
    flops_per_batch = flops_per_forward * BATCH_SIZE

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    best_epoch = 0
    perplexity_step = None
    global_step = 0

    epoch_times = []
    throughputs = []

    for epoch in range(EPOCHS):

        print(f"\nEpoch {epoch+1}/{EPOCHS}")

        train_loss, epoch_time, tokens_per_second = train_one_epoch(
            model, train_loader, optimizer, criterion
        )

        val_loss = evaluate(model, val_loader, criterion)

        epoch_times.append(epoch_time)
        throughputs.append(tokens_per_second)

        val_perplexity = math.exp(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1

        if perplexity_step is None and val_perplexity < PERPLEXITY_THRESHOLD:
            perplexity_step = global_step

        global_step += 1

        print("Train Loss:", train_loss)
        print("Validation Loss:", val_loss)

    test_loss = evaluate(model, test_loader, criterion)

    val_perplexity = math.exp(val_loss)
    test_perplexity = math.exp(test_loss)

    generalization_gap = train_loss - val_loss

    print("\n=================================================")
    print("MODEL: LSTM")
    print("=================================================")

    print("Total Parameters:", total_params)
    print("Trainable Parameters:", trainable_params)

    print("\nFinal Training Loss:", train_loss)
    print("Final Validation Loss:", val_loss)
    print("Final Test Loss:", test_loss)

    print("\nValidation Perplexity:", val_perplexity)
    print("Test Perplexity:", test_perplexity)

    print("\nGeneralization Gap:", generalization_gap)

    print("\nBest Validation Epoch:", best_epoch)
    print("Perplexity Threshold Step:", perplexity_step)

    print("\nTraining Time Per Epoch:", sum(epoch_times)/len(epoch_times))
    print("Tokens per Second:", sum(throughputs)/len(throughputs))

    print("Estimated FLOPs per Batch:", flops_per_batch)

    print("=================================================")


if __name__ == "__main__":
    main()

Using device: cuda
Using Colab cache for faster access to the 'tiny-shakespeare' dataset.
Vocabulary size: 65
Parameters: 420289

Epoch 1/5
Train Loss: 1.7085698622429442
Validation Loss: 1.6201005157302408

Epoch 2/5
Train Loss: 1.3652959016625752
Validation Loss: 1.5637410627860648

Epoch 3/5
Train Loss: 1.2850919365169045
Validation Loss: 1.5574098971544528

Epoch 4/5
Train Loss: 1.2367388283778094
Validation Loss: 1.5720750023336971

Epoch 5/5
Train Loss: 1.2007686293410684
Validation Loss: 1.5847711942943872

MODEL: LSTM
Total Parameters: 420289
Trainable Parameters: 420289

Final Training Loss: 1.2007686293410684
Final Validation Loss: 1.5847711942943872
Final Test Loss: 1.7150328808100277

Validation Perplexity: 4.878175096516695
Test Perplexity: 5.556858223357153

Generalization Gap: -0.38400256495331875

Best Validation Epoch: 3
Perplexity Threshold Step: 0

Training Time Per Epoch: 59.73039870262146
Tokens per Second: 916114.380368234
Estimated FLOPs per Batch: 13772029952
